In [1]:
%pip install torch
%pip install numpy
%pip install pandas
%pip install tqdm

  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   - -------------------------------------- 3.1/114.6 MB 16.8 MB/s eta 0:00:07
   --- ------------------------------------ 8.9/114.6 MB 22.1 MB/s eta 0:00:05
   ----- ---------------------------------- 14.4/114.6 MB 23.2 MB/s eta 0:00:05
   ------ --------------------------------- 19.9/114.6 MB 23.7 MB/s eta 0:00:04
   -------- ------------------------------- 23.1/114.6 MB 21.8 MB/s eta 0:00:05
   --------- ------------------------------ 27.5/114.6 MB 21.


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ------- -------------------------------- 2.4/12.3 MB 16.8 MB/s eta 0:00:01
   ----------------------- ---------------- 7.3/12.3 MB 20.6 MB/s eta 0:00:01
   ---------------------------------------  12.1/12.3 MB 21.6 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 20.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------------- -------------------------- 3.4/9.7 MB 22.3 MB/s eta 0:00:01
   ----------------------------------- ---- 8.7/9.7 MB 23.4 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 21.7 MB/s eta 0:00:00
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# 1. Define the Hierarchical Layer
class HierarchicalEncoder(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=512):
        super().__init__()
        # Project HuBERT features to a consistent hidden dimension
        self.projection = nn.Linear(input_dim, hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, 
            nhead=8, 
            dim_feedforward=1024, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

    def forward(self, x, mask=None):
        # x shape: [Batch, Max_Utts, Input_Dim]
        x = self.projection(x)
        # Apply self-attention across the sequence of utterances
        contextual_embs = self.transformer(x, src_key_padding_mask=mask)
        return contextual_embs

# 2. Dataset to Group Utterances by Dialogue
class MELDDialogueDataset(Dataset):
    def __init__(self, csv_path, npz_path):
        self.df = pd.read_csv(csv_path)
        self.features = np.load(npz_path, allow_pickle=True)
        # Group indices by Dialogue_ID to maintain hierarchy
        self.dialogue_groups = self.df.groupby('Dialogue_ID').groups

    def __len__(self):
        return len(self.dialogue_groups)

    def __getitem__(self, idx):
        dia_id = list(self.dialogue_groups.keys())[idx]
        indices = self.dialogue_groups[dia_id]
        
        # Pull the HuBERT features for all utterances in this dialogue
        dia_features = []
        for i in indices:
            feat_key = str(i)
            dia_features.append(self.features[feat_key])
        
        return torch.tensor(np.array(dia_features), dtype=torch.float32), dia_id

# 3. Custom Collate to handle varying dialogue lengths (Padding)
def collate_dialogues(batch):
    features, dia_ids = zip(*batch)
    lengths = [f.size(0) for f in features]
    max_len = max(lengths)
    
    # Pad sequences so they fit in a batch tensor
    padded_features = torch.zeros(len(features), max_len, features[0].size(1))
    mask = torch.ones(len(features), max_len, dtype=torch.bool)
    
    for i, f in enumerate(features):
        padded_features[i, :lengths[i], :] = f
        mask[i, :lengths[i]] = False # False means "do not mask"
        
    return padded_features, mask, dia_ids, lengths

# --- MAIN EXECUTION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize
dataset = MELDDialogueDataset('train_sent_emo.csv', 'meld_hubert_train.npz')
loader = DataLoader(dataset, batch_size=16, collate_fn=collate_dialogues)
model = HierarchicalEncoder(input_dim=768, hidden_dim=512).to(device)

model.eval()
hierarchical_results = {}

print("Generating Hierarchical Embeddings...")
with torch.no_grad():
    for feats, mask, ids, lens in tqdm(loader):
        feats, mask = feats.to(device), mask.to(device)
        
        # Generate the contextualized embeddings
        output = model(feats, mask=mask) # [Batch, Max_Len, Hidden_Dim]
        
        # Remove padding and save to dictionary
        for i, dia_id in enumerate(ids):
            actual_len = lens[i]
            # Store as [Num_Utterances, Hidden_Dim]
            hierarchical_results[str(dia_id)] = output[i, :actual_len, :].cpu().numpy()

# Save the new hierarchical bank
np.savez_compressed('meld_hierarchical_context_train.npz', **hierarchical_results)
print(f"Saved {len(hierarchical_results)} dialogue embeddings.")